# Get AfriSpeech data into your training notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfriSpeech/afrispeech-selector/blob/main/notebooks/afrispeech_selector.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/AfriSpeech/afrispeech-selector/blob/main/notebooks/afrispeech_selector.ipynb)

**Purpose:** drop-in cells for your own training notebook (Colab or Kaggle). They
install AfriSpeech Selector and pull your selected language(s) straight into the
running session, so your training script has the data without manual downloads.

It's clean read speech with aligned transcripts — a natural fit for **TTS**
(Option A). It also works for **ASR** (Option B), handy for supplementing
low-resource languages.

Use whatever runtime your **training** needs (a GPU for the actual model). On
**Kaggle**, turn **Internet ON** (`Settings → Internet`) so the data can download.

## Install (one cell)

In [ ]:
%pip install -q "afrispeech-selector @ git+https://github.com/AfriSpeech/afrispeech-selector.git"

# Audio backend check. datasets>=4 decodes audio via `torchcodec`, whose wheels are
# tied to a specific torch build — on some Colab/Kaggle runtimes the auto-installed
# one won't match the preinstalled torch. Fallback: pin datasets<4, which decodes
# (incl. mp3) via soundfile/librosa and needs no torchcodec.
import subprocess, sys
from importlib.metadata import version

def _torchcodec_ok():
    try:
        import torchcodec  # noqa: F401
        return True
    except Exception:
        return False

if int(version("datasets").split(".")[0]) >= 4 and not _torchcodec_ok():
    print("torchcodec missing/mismatched — trying to install a matching build…")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchcodec"])
    if not _torchcodec_ok():
        print("Falling back to datasets<4 (soundfile backend; no torchcodec needed).")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets<4", "soundfile", "librosa"])

print("Setup ready. torchcodec available:", _torchcodec_ok(), "| datasets:", version("datasets"))

## Option A — a local set for a TTS framework
Writes `wavs/` + a manifest into the session for Piper / VITS / MeloTTS / LJSpeech;
point your TTS data-prep at the folder. Transcripts are written verbatim.

In [ ]:
# --yes proceeds without prompting if a language has fewer hours than requested
!afrispeech-select --languages twi_twi --total-hours 5 --out data/twi --format ljspeech -y
!ls data/twi && head -2 data/twi/metadata.csv
# swap --format for piper / vits / melo (or combine: --format vits,melo)

## Option B — stream into ASR training (no copy)
Well suited to supplementing low-resource ASR. Hand the `IterableDataset` to your
🤗 `Trainer` / DataLoader — nothing is written to disk.

In [ ]:
from afrispeech_selector import stream_dataset

ds = stream_dataset(
    ["twi_twi"],                 # one or more languages (see afrispeech-select --list-langs)
    split="train",
    max_seconds=5 * 3600,         # ~5 h; or per_language=200 for a clip count
    target_sampling_rate=16000,   # min/max clip length default to 3-15s
)

# `ds` is a datasets.IterableDataset — plug it into your training loop:
for batch in ds.iter(batch_size=8):
    audio, text = batch["audio"], batch["text"]
    break  # <-- replace with your forward / loss step
print("streaming OK:", text[0][:80])

### ASR with a cached dataset (`load_from_disk`)
If you'd rather have random access (multiple epochs, shuffling) than a stream.

In [ ]:
!afrispeech-select --languages twi_twi --total-hours 5 --out data/twi --format disk --target-sr 16000 -y
from datasets import load_from_disk
ds = load_from_disk("data/twi").train_test_split(test_size=0.1)
print(ds)

### More than one language
Pass several names, or select across the dataset by strength — the budget is split
evenly per language:

```python
stream_dataset(["twi_twi", "ewe_ewe", "ga_gaa"], split="train", max_seconds=2 * 3600)
```
```bash
!afrispeech-select --top 10 --max-per-country 2 --total-hours 24 --out data/multi --format ljspeech -y
```

Output defaults to `data/<language>` if you omit `--out`. Full options and the
language list: https://github.com/AfriSpeech/afrispeech-selector